# 3. Select pairs, extract environments, and inspect forces
**Input:** your dump, explicit units, optional data file.
**Output:** candidate assessments, a finite pair/shell, source forces and optional exports.
**Modules:** lammpsio → geom_utils / pair_selection → visualization / geometry.

Automatic selection, angle assessment and plane overlays are **CPD/dienophile-specific**. For other chemistry, provide two complete molecule-ID lists and a relative lattice shift. This supports geometric extraction, not general reaction classification. No saved reports are required.

Set `RUN_DIR` to your simulation folder. Leave `INPUT_FILE`, `DUMP_FILE`, and `DATA_FILE` as `None` for automatic discovery. Files directly in the folder take priority; subfolders are searched if necessary. If several files match, set the corresponding filename override. Keep units and other physical settings explicit.

In [ ]:
from pathlib import Path
from io import StringIO
from dataclasses import asdict
import json
import numpy as np
import pandas as pd
from IPython.display import display
from ase.io import read, write
from MartiniFuncs.lammpsio import read_lammps_frame
from MartiniFuncs.trajectory_analysis import resolve_run_files
from MartiniFuncs import geom_utils, md_statistics
from MartiniFuncs.pair_selection import PairCriteria, assess_prereactive_pairs
from MartiniFuncs.visualization import plot_cluster, plot_pair_geometry
from MartiniFuncs.geometry import align_geometry

RUN_DIR = Path(r"/path/to/simulation")
INPUT_FILE = None  # optional filename override; None discovers automatically
DUMP_FILE = None
DATA_FILE = None
UNITS = "metal"
TIMESTEP_PS = 0.0005
TYPE_SYMBOLS = None
FRAME_INDEX = 0
BOND_SCALE = 1.3
REACTIVE_RADIUS = 3.25
SHELL_RADIUS = 3.0
CRITERIA = PairCriteria()  # optional maxima for angles, forming distance, imbalance
SELECTED_CANDIDATE = 0
MANUAL_PAIR_IDS = None  # ([persistent IDs of molecule A], [IDs of molecule B])
MANUAL_SHIFT = (0, 0, 0)  # integer lattice translation of A relative to B
REFERENCE_EXTXYZ = None  # optional same-identity finite pair
OUTPUT_DIR = RUN_DIR / "analysis"
WRITE_FILES = False
OVERWRITE = False


In [ ]:
files = resolve_run_files(RUN_DIR, input_file=INPUT_FILE, dump_file=DUMP_FILE, data_file=DATA_FILE, require_input=False)
print("Selected files:", files)
frame = read_lammps_frame(files["dump"], index=FRAME_INDEX, units=UNITS,
                          input_datafile=files["data"], type_symbols=TYPE_SYMBOLS)
atoms = frame.atoms
print(f"Frame {frame.index}; step {frame.timestep}; time {frame.timestep*TIMESTEP_PS:g} ps")


## Assess candidates
The distance screen uses CPD COM to reactive-C=C midpoint. PairCriteria adds optional upper limits; every enabled criterion must pass. Axis angle compares reactive axes; plane angle compares fitted normals; approach angle compares site separation with the diene normal. All angles are unsigned 0–90 degrees. Approach is position, not motion.

`forming_bonds` contains proposed persistent-ID pairings. `forming_bonds_local` indexes the extracted pair; `forming_bonds_frame_local` indexes the full frame. These do not establish bond formation, stereochemistry or a transition state.

In [ ]:
if MANUAL_PAIR_IDS is None:
    pairs, shifts = geom_utils.get_all_prereactive_pairs(atoms, scale=BOND_SCALE, reactive_radius=REACTIVE_RADIUS)
    assessments = assess_prereactive_pairs(atoms, pairs, shifts, criteria=CRITERIA, scale=BOND_SCALE)
    display(pd.DataFrame([
        {"candidate":i, "accepted":a.accepted, "reasons":"; ".join(a.reasons),
         **(asdict(a.geometry) if a.geometry is not None else {})}
        for i,a in enumerate(assessments)]))
else:
    pairs, shifts, assessments = [MANUAL_PAIR_IDS], [MANUAL_SHIFT], None
if not 0 <= SELECTED_CANDIDATE < len(pairs):
    raise ValueError("No such candidate: inspect the table or supply a manual pair")

def custom_check(atoms, pair_ids, shift, assessment):
    return True  # replace with your own test; no callback API is assumed

assessment = assessments[SELECTED_CANDIDATE] if assessments is not None else None
if assessment is not None and not assessment.accepted:
    raise ValueError(f"Selected candidate failed: {assessment.reasons}")
if not custom_check(atoms, pairs[SELECTED_CANDIDATE], shifts[SELECTED_CANDIDATE], assessment):
    raise ValueError("Selected candidate failed your custom check")


## Extract whole molecules and check ordering
The shell cutoff is atom-to-pair distance. Every touched molecule is included completely and placed in a consistent periodic image. The result is finite and nonperiodic.

Legacy role labels CPD/MAL denote first/second molecules even in manual mode; they are not chemical identification for other systems.

In [ ]:
clusters = geom_utils.get_pair_shell_clusters(
    atoms, [pairs[SELECTED_CANDIDATE]], [shifts[SELECTED_CANDIDATE]],
    shell_radius=SHELL_RADIUS, scale=BOND_SCALE)
records = geom_utils.get_cluster_positions_and_forces(atoms, frame, clusters)
buffer = StringIO()
geom_utils.pair_data_to_extxyz(records, buffer)
buffer.seek(0)
cluster = read(buffer, format="extxyz")
pair = cluster[cluster.arrays["pair_role"] != "shell"]
lookup = {int(i): row for row,i in enumerate(atoms.arrays["global_index"])}
source_rows = [lookup[int(i)] for i in cluster.arrays["global_index"]]
if "lammps_forces" in cluster.arrays:
    np.testing.assert_allclose(cluster.arrays["lammps_forces"], atoms.arrays["lammps_forces"][source_rows], atol=1e-8)
display(pd.DataFrame({"pair_row":np.arange(len(pair)), "global_index":pair.arrays["global_index"],
                      "role":pair.arrays["pair_role"], "element":pair.get_chemical_symbols()}))
print("Pair / shell atoms:", len(pair), len(cluster)-len(pair))


In [ ]:
display(plot_cluster(cluster))
display(plot_cluster(pair, forces="lammps_forces" in pair.arrays))
if assessment is not None and assessment.geometry is not None:
    display(plot_pair_geometry(pair, disk_radius=2., transparency=.5, scale=BOND_SCALE))
if REFERENCE_EXTXYZ is not None:
    aligned, rmsd, transform = align_geometry(pair, read(REFERENCE_EXTXYZ))
    print("Aligned RMSD (angstrom):", rmsd)
    display(plot_cluster(aligned, forces="lammps_forces" in aligned.arrays))


ASE positions are angstrom; conversion to Psience Molecule uses Bohr. Plots and overlays use angstrom. Bonds retain Psience defaults. Alignment rotates geometry and force vectors together; original coordinates remain the export source.

In [ ]:
first_ids, second_ids = pairs[SELECTED_CANDIDATE]
first_center = md_statistics.get_molecule_center_of_mass(pair, first_ids, positions_unwrapped=True)
second_center = md_statistics.get_molecule_center_of_mass(pair, second_ids, positions_unwrapped=True)
if "momenta" in pair.arrays:
    print("Relative COM velocity (angstrom/ps):",
          md_statistics.get_pair_relative_velocity(pair, first_ids, second_ids))
if "lammps_forces" in pair.arrays:
    print("Net-force projections (eV/angstrom):",
          md_statistics.get_pair_force_projections(pair, first_ids, second_ids, direction=second_center-first_center))


## Export
Enable WRITE_FILES to create your own inputs for notebook 07. These are total source MD forces, not validated external loads.

In [ ]:
if WRITE_FILES:
    paths = [OUTPUT_DIR/name for name in ("pair.extxyz","cluster.extxyz","selection.json")]
    if not OVERWRITE and any(p.exists() for p in paths):
        raise FileExistsError("Choose a new OUTPUT_DIR or enable OVERWRITE")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    write(paths[0], pair)
    write(paths[1], cluster)
    provenance = {"dump":str(files["dump"]), "frame":frame.index, "step":frame.timestep,
                  "units":UNITS, "timestep_ps":TIMESTEP_PS, "shell_radius_angstrom":SHELL_RADIUS,
                  "pair_global_ids":pair.arrays["global_index"].tolist(),
                  "geometry":asdict(assessment.geometry) if assessment and assessment.geometry else None}
    paths[2].write_text(json.dumps(provenance, indent=2, default=lambda x: int(x))+chr(10))
    print("Saved:", OUTPUT_DIR)
